# Life Insurance Portfolio Simulation and Mortality Experience Analysis

## Overview

In this notebook we simulate a life insurance portfolio and analyze mortality experience using Canadian life table data.

The objective is to demonstrate how to estimate expected deaths using mortality rates and compare them with observed claims. This process, commonly known as **Actual vs Expected (A/E) analysis**, is widely used in life insurance pricing, risk monitoring, and experience studies.

## Objectives

1. Simulate a realistic life insurance policy dataset
2. Assign mortality probabilities to each policyholder using the Canadian life table
3. Calculate expected deaths using actuarial mortality rates
4. Simulate actual claims experience
5. Compare actual vs expected mortality outcomes

## Key Actuarial Concept

Expected deaths are calculated using:

Expected Deaths = Exposure × Mortality Rate (qx)

This method forms the basis of mortality experience analysis used by life insurers and reinsurers to monitor risk and update mortality assumptions.

In [2]:
# Load clean table and generate data

import numpy as np
import pandas as pd
mortality_table = pd.read_csv("../data/processed/mortality_table.csv")
mortality_table.head()

,age,survivors_lx,deaths_dx,mortality_rate_qx,central_death_rate_mx,survival_probability_px,person_years_lived_Lx,total_future_years_Tx,life_expectancy_ex,std_error_life_expectancy,sex
0,0,100000,489,0.00489,0.00031,0.99511,99551,8010812,80.11,0.07,male
1,1,99511,26,0.00026,0.00007,0.99974,99500,7911262,79.50,0.06,male
2,2,99485,18,0.00019,0.00006,0.99981,99473,7811762,78.52,0.06,male
3,3,99467,14,0.00014,0.00005,0.99986,99459,7712289,77.54,0.06,male
4,4,99453,11,0.00011,0.00005,0.99989,99446,7612829,76.55,0.06,male


In [12]:
# Generating synthetic insurance portfolio

np.random.seed(42)
portfolio_size = 10000
policy_id = np.arange(1, portfolio_size + 1) #generate policy IDs
ages = np.random.randint(20, 81, portfolio_size) #generate ages
sex = np.random.choice(["male", "female"], portfolio_size) #assign gender
policy_duration = np.random.randint(1, 31, portfolio_size) #generate policy duration
sum_insured = np.random.randint(50000, 1000000, portfolio_size) #generate sum insured
portfolio = pd.DataFrame({       #build portfolio data frame
    "policy_id": policy_id,
    "age": ages,
    "sex": sex,
    "policy_duration": policy_duration,
    "sum_insured": sum_insured
})
portfolio

,policy_id,age,sex,policy_duration,sum_insured
0,1,58,female,18,190707
1,2,71,female,21,725424
2,3,48,female,1,584228
3,4,34,male,24,871601
4,5,62,female,21,390791
...,...,...,...,...,...
9995,9996,34,male,10,371130
9996,9997,46,male,25,827499
9997,9998,23,male,13,710667
9998,9999,35,female,22,849336


In [8]:
# Attach Mortality Rates

portfolio = portfolio.merge(mortality_table,on=["age", "sex"],how="left")
portfolio

,policy_id,age,sex,policy_duration,sum_insured,survivors_lx_x,deaths_dx_x,mortality_rate_qx_x,central_death_rate_mx_x,survival_probability_px_x,...,std_error_life_expectancy_y,survivors_lx,deaths_dx,mortality_rate_qx,central_death_rate_mx,survival_probability_px,person_years_lived_Lx,total_future_years_Tx,life_expectancy_ex,std_error_life_expectancy
0,1,58,female,18,190707,95322,343,0.00360,0.00022,0.99640,...,0.05,95322,343,0.00360,0.00022,0.99640,95150,2712718,28.46,0.05
1,2,71,female,21,725424,87309,1098,0.01257,0.00050,0.98743,...,0.04,87309,1098,0.01257,0.00050,0.98743,86760,1515782,17.36,0.04
2,3,48,female,1,584228,97557,152,0.00155,0.00015,0.99845,...,0.05,97557,152,0.00155,0.00015,0.99845,97482,3678599,37.71,0.05
3,4,34,male,24,871601,97902,111,0.00113,0.00013,0.99887,...,0.05,97902,111,0.00113,0.00013,0.99887,97847,4643094,47.43,0.05
4,5,62,female,21,390791,93752,486,0.00519,0.00028,0.99481,...,0.05,93752,486,0.00519,0.00028,0.99481,93509,2334400,24.90,0.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,34,male,10,371130,97902,111,0.00113,0.00013,0.99887,...,0.05,97902,111,0.00113,0.00013,0.99887,97847,4643094,47.43,0.05
9996,9997,46,male,25,827499,96227,202,0.00210,0.00019,0.99790,...,0.05,96227,202,0.00210,0.00019,0.99790,96126,3477318,36.14,0.05
9997,9998,23,male,13,710667,98928,80,0.00081,0.00011,0.99919,...,0.06,98928,80,0.00081,0.00011,0.99919,98888,5725980,57.88,0.06
9998,9999,35,female,22,849336,98739,59,0.00059,0.00009,0.99941,...,0.05,98739,59,0.00059,0.00009,0.99941,98710,4955768,50.19,0.05


In [10]:
portfolio["expected_death_probability"] = portfolio["mortality_rate_qx_x"] # Calculate Expected Moratlity
portfolio["actual_death"] = np.random.binomial(1,portfolio["mortality_rate_qx_x"]) # Simulate Actual Death Claims
portfolio["claim_amount"] = portfolio["actual_death"] * portfolio["sum_insured"] # Calculate claim amount

In [11]:
# Portfolio Summary

total_policies = len(portfolio)
expected_deaths = portfolio["expected_death_probability"].sum()
actual_deaths = portfolio["actual_death"].sum()
ae_ratio = actual_deaths / expected_deaths

print("Total Policies:", total_policies)
print("Expected Deaths:", round(expected_deaths,2))
print("Actual Deaths:", actual_deaths)
print("A/E Ratio:", round(ae_ratio,2))

Total Policies: 10000
Expected Deaths: 72.48
Actual Deaths: 71
A/E Ratio: 0.98
